# AIDE-TIHM Notebook 2: Longitudinal In-Home IoT Dementia Telemonitoring

This notebook demonstrates continuous telemetry monitoring for People Living with Dementia (PLWD) using the TIHM open dataset schema (Palermo et al., Scientific Data 2023).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../src"))

from data_preprocessing import load_or_create_datasets
from feature_engineering import extract_tihm_monitoring_features
from tihm_event_model import split_tihm_participant_independent, train_and_evaluate_tihm_models


## 1. Load TIHM In-Home Telemetry Dataset


In [ ]:
_, tihm_df = load_or_create_datasets(data_dir="../data")
print("TIHM Dataset Shape:", tihm_df.shape)
print("Monitored Participants:", tihm_df['patient_id'].nunique())
print("Total Participant-Days:", len(tihm_df))
tihm_df.head()


## 2. Participant-Independent Splitting


In [ ]:
train_t, test_t = split_tihm_participant_independent(tihm_df, patient_col='patient_id', test_size=0.20)
print(f"Train Cohort: {train_t['patient_id'].nunique()} participants, {len(train_t)} days | Event Rate: {train_t['health_event_flag'].mean():.1%}")
print(f"Test Cohort:  {test_t['patient_id'].nunique()} participants, {len(test_t)} days | Event Rate: {test_t['health_event_flag'].mean():.1%}")


## 3. Extract Longitudinal Sensor Features & Baseline Deviations


In [ ]:
X_train, y_train, feature_cols, scaler, _ = extract_tihm_monitoring_features(train_t, is_training=True)
X_test, y_test, _, _, _ = extract_tihm_monitoring_features(test_t, is_training=False, scaler=scaler)
print(f"Engineered IoT Features ({len(feature_cols)}):", feature_cols[:10])


## 4. Train & Compare Telemonitoring Event Detectors


In [ ]:
results = train_and_evaluate_tihm_models(X_train, y_train, X_test, y_test)
print("Best Telemonitoring Detector:", results['best_model_name'])
for name, m in results['metrics'].items():
    print(f"\nDetector: {name}")
    print(f"  PR-AUC:  {m['pr_auc']:.4f}")
    print(f"  ROC-AUC: {m['roc_auc']:.4f}")
    print(f"  Recall:  {m['recall']:.4f}")
    print(f"  F1:      {m['f1_score']:.4f}")


## 5. Visualizing a Single Patient's Trajectory & Early Warnings


In [ ]:
sample_pid = test_t['patient_id'].iloc[0]
df_sample = test_t[test_t['patient_id'] == sample_pid].sort_values('date')

plt.figure(figsize=(12, 6))
plt.plot(pd.to_datetime(df_sample['date']), df_sample['night_bathroom_trips'], marker='o', label='Night Bathroom Trips', color='red')
plt.plot(pd.to_datetime(df_sample['date']), df_sample['sleep_duration_hours'], marker='s', label='Sleep Hours', color='blue')
plt.title(f"Telemetry Trajectory: Participant {sample_pid}", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Sensor Count / Hours")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
